Завдання 1

Клас менеджера файлового контексту

Створіть власний клас, який може поводитися як вбудована функція 'open'. Також вам потрібно розширити його функціональність за допомогою лічильника та логування. Зверніть особливу увагу на реалізацію методу '__exit__', який має відповідати всім вимогам до менеджерів контексту, згаданим тут:

Типи менеджерів контексту 

Оператор with

In [3]:
import logging

class FileContextManager:
    counter = 0  # лічильник відкритих файлів

    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode
        self.file = None

        # налаштування логування
        logging.basicConfig(filename="file_manager.log", level=logging.INFO)

    def __enter__(self):
        FileContextManager.counter += 1
        logging.info(f"Відкриття файлу: {self.filename}, режим: {self.mode}")
        self.file = open(self.filename, self.mode, encoding="utf-8")
        return self.file

    def __exit__(self, exc_type, exc_value, traceback):
        if self.file:
            self.file.close()
            logging.info(f"Закриття файлу: {self.filename}")
        FileContextManager.counter -= 1

    
        if exc_type:
            logging.error(f"Помилка: {exc_type}, {exc_value}")
            
            return False
        return True


Написання тестів для менеджера контексту r

Візьміть свою реалізацію класу менеджера контексту із Завдання 1 та напишіть для неї тести. Намагайтеся охопити якомога більше випадків використання, позитивних, коли файл існує і все працює належним чином. А також пишіть тести, коли ваш клас викликає помилки або у вас є помилки в наборі контекстів виконання.



In [2]:
import unittest
import os

class TestFileContextManager(unittest.TestCase):
    def setUp(self):
        self.test_file = "test.txt"

    def tearDown(self):
        if os.path.exists(self.test_file):
            os.remove(self.test_file)

    def test_write_and_read(self):
        # запис у файл
        with FileContextManager(self.test_file, "w") as f:
            f.write("Hello, World!")

        # читання з файлу
        with FileContextManager(self.test_file, "r") as f:
            content = f.read()
        self.assertEqual(content, "Hello, World!")

    def test_counter_increments(self):
        start_count = FileContextManager.counter
        with FileContextManager(self.test_file, "w") as f:
            f.write("test")
        self.assertEqual(FileContextManager.counter, start_count)

    def test_exception_handling(self):
        # перевірка, що виняток не пригнічується
        with self.assertRaises(ValueError):
            with FileContextManager(self.test_file, "w") as f:
                raise ValueError("Тестова помилка")

    def test_file_closed_after_exit(self):
        manager = FileContextManager(self.test_file, "w")
        with manager as f:
            f.write("data")
        self.assertTrue(f.closed)
